# Lacuna 2023: every current number, from DuckDB to the consensus

This notebook rebuilds the current Lacuna results (Zimbabwe, 2023 season) from the two team databases, with the code
that made them. It covers our experiments (baselines, time features, step-up ladder, Presto controls, fold 0) and
Felix's experiments (merge, transfer diagnosis, weather, aggregation), and it shows why the two sets of numbers differ.

**Run it** from the `notebooks/` folder with the project `.venv` kernel. It needs `data/cropyield.duckdb` (v1) and
`data/v2/cropyield.duckdb` (v2). It opens both read-only, writes nothing, and takes a few minutes. Each recomputed
number that has a committed counterpart is checked against it with an `assert`, so a failed cell means the numbers in
the repository and the code no longer agree. The switch chain (section 6.2) and the protocol grid (section 6.3) are
new computations: their end points are checked against Felix's and our committed files, and the last cell checks
every number quoted in "The answer".

All scores are on the same 1,504 Lacuna fields (folds 1–4 of `s1_cc2km`) unless a cell says fold 0 (376 fields).
RMSE, MAE and bias are in t/ha; bias is prediction minus yield; R² is 1 − SSE/SST over the scored fields of this one
cohort; "rank" is the Spearman correlation inside 2 km clusters after each cluster mean is removed.

## The answer (8 Nov 2026)

All numbers below are on the same 1,504 Lacuna 2023 fields (folds 1–4) unless they say fold 0 (376 fields). The last
code cell checks every number in this list against the values the notebook computes.

1. **Monthly satellite bands are the best inputs we have.** Ridge on the monthly band curve (M2) reaches R² 0.322
   with our protocol and 0.290 with Felix's, against 0.191 for the 10 season indices (A1) and 0.127 for coordinates
   alone (R1). On fold 0, M2 reaches R² 0.396 and an RMSE 0.246 t/ha lower than A1 (interval −0.363 to −0.089).
2. **Our monthly model and Felix's monthly model are equally good.** Inside one protocol their RMSE difference
   stays inside its paired interval: our M2 (70 inputs) minus his monthly Ridge (122 inputs) is +0.008 t/ha
   (−0.023 to +0.040) with our protocol and −0.021 (−0.055 to +0.014) with his (R² 0.322 against 0.329, and 0.290
   against 0.272). The gap between "our 0.32" and "his 0.27" is the protocol: keeping fold 0 out of training
   costs M2 0.028 to 0.035 t/ha and his model 0.036 to 0.055 t/ha, while his head changes RMSE by at most 0.020 and
   the database by at most 0.013 (section 6.3).
3. **Presto adds nothing over its own inputs.** Frozen Presto (P0, R² 0.302) is not better than Ridge on the raw
   monthly inputs (X1, R² 0.328): P0 minus X1 is +0.031 t/ha RMSE (−0.016 to +0.078). Fine-tuned Presto (P1) reaches
   R² 0.307.
   Codex's debug runs (section 4.5, 128 training fields) confirm that fine-tuning works but overfits after
   6–14 epochs at every dropout level; dropout only delays it, and the best validation RMSE is 1.58–1.59 at
   every level.
4. **No season summary, window or ladder step beats A1 on the rank inside clusters.** After the Holm correction no
   ladder row is below 0.05; the smallest p-value is 0.222 (frozen Presto with a Bayesian ridge head; 0.288 with
   the Ridge head). Felix's December–March band mean lowers RMSE by 0.044 t/ha
   (−0.081 to −0.009) against the whole-season band mean, but that is a gain over another season mean, not over A1.
5. **Most of the skill is between villages.** M2 predicts the 2 km cluster means with R² 0.502 and the fields inside
   a cluster with R² 0.096 (A1: 0.299 and 0.055). The rank inside clusters is 0.299 for A1 and 0.342 for M2.
6. **Adding Zimbabwe 2024 gives no better model than the best Lacuna-only one.** For the index Ridge, adaptive
   pooling changes RMSE by −0.0036 t/ha (−0.0084 to +0.0007); naive pooling is much worse (RMSE 2.086). Monthly
   trees with 2024 are not clearly different from Lacuna-only trees: +0.018 t/ha (−0.007 to +0.045). One exception:
   with 4 weather features, the Ridge shared with 2024 beats its Lacuna-only twin by 0.086 t/ha (−0.140 to −0.041),
   but that twin is weak (RMSE 1.728), and the pooled arm does not beat the Lacuna-only trees (item 7).
7. **Weather is not established.** Felix's best weather arm (4 weather features, Ridge shared with 2024, RMSE 1.643)
   against Lacuna-only monthly trees (1.659): −0.017 t/ha (−0.058 to +0.027).
8. **Fold 0 has the same order, and every model predicts too low there.** On fold 0, M2, M2-pre and X1 have the lowest
   RMSE, and no arm beats A1 on the rank after Holm. The training mean misses fold 0 by −0.562 t/ha: its fields yield
   more than folds 1–4.

**Felix's numbers.** His paper files (`docs/paper/`, 7 Nov) quote his v1 runs; section 6.1 gives the rule-compliant
v2 value of each headline row. His Lacuna-only Ridge (RMSE 1.797) is our A1 (1.785) after three switches: the v1
database (+0.001), keeping fold 0 out of training (+0.008) and his smaller alpha grid (+0.003); section 6.2 reproduces
his number to six decimals with our code.

**Not settled:** whether fold 0 trains in development (our protocol) or not (his). Both are written in
`docs/EVALUATION.md`. Pick one before the report puts our arms and his arms in one table.

## 1. Setup and the two databases

The two databases must equal their pins in `configs/`. Data v2 differs from v1 in Malawi 2022, in cloudy months that
v2 filled from more candidate scenes, and in holding no Presto or MOSAIKS rows (our P0 embeddings are recomputed from
v2 inputs in `experiments/presto_finetune/`; only ladder step S12 reads MOSAIKS from v1). For Lacuna the labels, rows
and folds are the same.

In [1]:
import hashlib
import importlib.util
import json
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO / "src"))


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    loaded = importlib.util.module_from_spec(spec)
    sys.modules[name] = loaded
    spec.loader.exec_module(loaded)
    return loaded


step8 = load_module("baseline_table", REPO / "scripts" / "baselines" / "baseline_table.py")
tf = load_module("time_features", REPO / "experiments" / "time_features" / "time_features.py")
from cropyield.baselines.models import ridge_cv
from cropyield.data.xy import load_xy
from cropyield.eval.scores import bias, mae, r2, rank_inside_clusters, rmse

COHORT = "zimbabwe_lacuna_2022_2023"
V1, V2 = REPO / "data", REPO / "data" / "v2"
pd.set_option("display.precision", 4)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 90)


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


pins = []
for version, folder in (("v1", V1), ("v2", V2)):
    pin = json.loads((REPO / "configs" / f"team_database_{version}.json").read_text(encoding="utf-8"))
    path = folder / "cropyield.duckdb"
    pins.append({"database": version, "file": str(path.relative_to(REPO)), "bytes": path.stat().st_size,
                 "sha256": sha256(path)[:12], "pin": pin["sha256"][:12], "matches_pin": sha256(path) == pin["sha256"]})
pins = pd.DataFrame(pins)
assert pins["matches_pin"].all(), "a database differs from its pin"
pins

,database,file,bytes,sha256,pin,matches_pin
0,v1,data\cropyield.duckdb,872689664,e192b9d6021f,e192b9d6021f,True
1,v2,data\v2\cropyield.duckdb,22818816,5f36d91bfafa,5f36d91bfafa,True


## 2. The Lacuna data: labels, model rows, folds

`labels_v2` holds every Lacuna crop cut that passed the source checks (2,266). `include_primary` keeps the 1,880
rainfed, pure-stand 2023 fields; Felix's handoff gives the same 1,880. Every model row needs at least 4 clear months.

In [2]:
labels = pd.read_parquet(V2 / "processed" / "labels_v2.parquet").set_index("obs_id")
lacuna = labels[labels["cohort"] == COHORT]
funnel = (lacuna.assign(exclusion_reason=lacuna["exclusion_reason"].replace("", "kept (include_primary)"))
          .groupby(["season_year", "exclusion_reason"]).size().rename("rows").reset_index())
print(f"labels_v2 Lacuna rows: {len(lacuna):,}; include_primary: {int(lacuna['include_primary'].sum()):,}; "
      f"zero yields kept: {int((lacuna['include_primary'] & lacuna['is_zero']).sum())}")
funnel

labels_v2 Lacuna rows: 2,266; include_primary: 1,880; zero yields kept: 28


,season_year,exclusion_reason,rows
0,2022,dropped_year_2022_imagery_gap,169
1,2023,kept (include_primary),1880
2,2023,lacuna_intercropped,212
3,2023,lacuna_irrigated,5


In [3]:
expected, pre_months, _ = tf.COHORTS[COHORT]
X_a1, y, fields = step8.cohort_rows(COHORT, V2, labels, expected, "labels_v2")
folds = fields.assign(y=y).groupby("fold").agg(fields=("y", "size"), clusters=("group_id", "nunique"),
                                                mean_yield_t_ha=("y", "mean"))
print(f"model rows: {len(y):,}; folds 1-4 (scored in development): {int((fields['fold'] != '0').sum()):,}; "
      f"fold 0: {int((fields['fold'] == '0').sum())}")
folds

model rows: 1,880; folds 1-4 (scored in development): 1,504; fold 0: 376


,fields,clusters,mean_yield_t_ha
fold,,,
0,376,57,3.7955
1,376,58,3.0262
2,376,58,3.3162
3,376,58,3.1939
4,376,58,3.3980


In [4]:
v2 = duckdb.connect(str(V2 / "cropyield.duckdb"), read_only=True)
v1 = duckdb.connect(str(V1 / "cropyield.duckdb"), read_only=True)
tables = v2.sql("SELECT table_name, table_type FROM information_schema.tables ORDER BY table_name").df()
primary = list(y.index)


def lacuna_rows(con, table):
    if table in ("build_info",):
        return None
    return con.execute(f"SELECT count(*) FROM {table} WHERE obs_id IN (SELECT unnest(?))", [primary]).fetchone()[0]


tables["lacuna_model_rows_v2"] = [lacuna_rows(v2, t) for t in tables["table_name"]]
tables["lacuna_model_rows_v1"] = [lacuna_rows(v1, t) for t in tables["table_name"]]
tables

,table_name,table_type,lacuna_model_rows_v2,lacuna_model_rows_v1
0,build_info,BASE TABLE,NaN,NaN
1,coverage,BASE TABLE,16920.0,16920.0
2,downloads,BASE TABLE,1880.0,1880.0
3,labels,BASE TABLE,1880.0,1880.0
4,master,VIEW,1880.0,1880.0
5,monthly,VIEW,16920.0,16920.0
6,monthly_bands,BASE TABLE,16920.0,16920.0
7,mosaiks_monthly,BASE TABLE,0.0,12010.0
8,mosaiks_season,BASE TABLE,0.0,1880.0
9,presto_embeddings,BASE TABLE,0.0,1880.0


In [5]:
coverage = []
for version, con in (("v1", v1), ("v2", v2)):
    state = con.execute("""SELECT month, coverage_state, count(*) AS n FROM monthly
                           WHERE obs_id IN (SELECT unnest(?)) GROUP BY ALL""", [primary]).df()
    state = state.pivot(index="month", columns="coverage_state", values="n").fillna(0).astype(int)
    coverage.append(state.assign(clear_share=(state.get("clear", 0) / len(primary)).round(3)).add_prefix(f"{version}_"))
coverage = pd.concat(coverage, axis=1)
print("October and November 2022 have no accepted scene for any Lacuna field, so the usable window is Dec-Jun.")
coverage

October and November 2022 have no accepted scene for any Lacuna field, so the usable window is Dec-Jun.


coverage_state,v1_clear,v1_cloudy,v1_no_scene,v1_clear_share,v2_clear,v2_cloudy,v2_no_scene,v2_clear_share
month,,,,,,,,
2022-10,0,0,1880,0.000,0,0,1880,0.000
2022-11,0,0,1880,0.000,0,0,1880,0.000
2022-12,1791,89,0,0.953,1835,45,0,0.976
2023-01,1502,378,0,0.799,1549,331,0,0.824
2023-02,1390,490,0,0.739,1421,459,0,0.756
2023-03,1687,193,0,0.897,1770,110,0,0.941
2023-04,1880,0,0,1.000,1880,0,0,1.000
2023-05,1880,0,0,1.000,1880,0,0,1.000
2023-06,1880,0,0,1.000,1880,0,0,1.000


In [6]:
print("A1 inputs: the 10 season indices of the database (max and mean of 5 vegetation indices over the clear months)")
display(X_a1.head(3))
print("One field in the monthly view (v2): the inputs of the monthly curves")
v2.execute("SELECT month, coverage_state, b02, b04, b08, b11, ndvi, gcvi FROM monthly WHERE obs_id = ? ORDER BY month",
           [primary[0]]).df()

A1 inputs: the 10 season indices of the database (max and mean of 5 vegetation indices over the clear months)
One field in the monthly view (v2): the inputs of the monthly curves


,ndvi_season_max,ndvi_season_mean,gcvi_season_max,gcvi_season_mean,mtci_season_max,mtci_season_mean,ndvi705_season_max,ndvi705_season_mean,ndvi740_season_max,ndvi740_season_mean
obs_id,,,,,,,,,,
zimbabwe:0002e51c-5124-4614-bb1b-1c1ae307de80,0.5969,0.4581,3.1957,2.3423,2.0294,1.3951,0.3337,0.2206,0.0483,0.0068
zimbabwe:002b9802-b084-47a9-9ba6-0e87a828b826,0.8443,0.4655,6.1009,2.6903,8.7620,3.3754,0.6595,0.3448,0.1666,0.0930
zimbabwe:0052277a-e7f7-4964-b02b-f985f6f56a4e,0.5198,0.2966,2.1730,1.3239,19.4695,6.0581,0.3597,0.2148,0.0999,0.0803


,month,coverage_state,b02,b04,b08,b11,ndvi,gcvi
0,2022-10,no_scene,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-11,no_scene,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-12,clear,0.0522,0.0682,0.2702,0.3456,0.5969,2.6712
3,2023-01,clear,0.0499,0.0949,0.2232,0.3494,0.4033,1.8837
4,2023-02,clear,0.0451,0.0850,0.2066,0.2867,0.4170,2.3323
5,2023-03,clear,0.0319,0.0602,0.1917,0.2778,0.5220,2.6724
6,2023-04,clear,0.0396,0.0586,0.2316,0.2546,0.5961,3.1957
7,2023-05,clear,0.0493,0.0974,0.2208,0.2909,0.3878,2.1543
8,2023-06,clear,0.0603,0.1036,0.1855,0.3154,0.2833,1.4866


## 3. Two protocols

Both teams score the same 1,504 fields with the same frozen folds and the same features code style, but not the
same way. This one difference explains most of the gap between our numbers and Felix's (section 6).

| | Protocol A (ours) | Protocol B (Felix) |
|---|---|---|
| Training rows for a test fold | the other 4 folds, **fold 0 included** (1,504 fields) | the other 3 development folds, **fold 0 never** (1,128 fields) |
| Inner tuning | `GroupKFold` 5, 17 alphas 1e-3 to 1e5 | `GroupKFold` 3, 6 alphas 0.01 to 1000, predictions clipped at 0 |
| Database | v2 (`data/v2/cropyield.duckdb`) | v1 (`data/cropyield.duckdb`) |
| Missing months | interpolated inside each field's own series | training-fold median plus missing-month flags |
| Written in | `docs/EVALUATION.md` line 61 ("fold-0 fields still train") | `docs/EVALUATION.md` line 7 ("fit on three folds, predict the fourth") |

The team has not yet picked one. Fold 0 itself is scored only by `experiments/fold0/score_fold0.py`: every model
trains on folds 1–4 and predicts fold 0, and each look is logged.

## 4. Our numbers, recomputed (protocol A, folds 1–4)

### 4.1 Baselines: training mean (R0), coordinates (R1) and the 10 season indices (A1)

In [7]:
scored = (fields["fold"] != "0").to_numpy()
yt = y.to_numpy()[scored]
cluster_years = (fields["group_id"] + "|" + fields["season_year"].astype(str)).to_numpy()


def scores(truth, pred, clusters):
    return {"rmse": rmse(truth, pred), "mae": mae(truth, pred), "bias": bias(truth, pred), "r2": r2(truth, pred),
            "rank": rank_inside_clusters(truth, pred, clusters, step8.MIN_CLUSTER_FIELDS)}


dev = step8.out_of_fold(X_a1, y, fields, ("0",))
ours = pd.DataFrame({arm: scores(yt, dev[arm][scored], cluster_years[scored]) for arm in ("R0", "R1", "A1")}).T
saved = (pd.read_csv(REPO / "results" / "v2" / "baseline_scores.csv").query("cohort == @COHORT")
         .set_index(["arm", "metric"])["value"])
for arm in ("R0", "R1", "A1"):
    for column, metric in (("rmse", "rmse_t_ha"), ("mae", "mae_t_ha"), ("bias", "bias_t_ha"), ("r2", "r2"),
                           ("rank", "rank_inside_clusters")):
        if not (np.isnan(ours.loc[arm, column]) and np.isnan(saved[arm, metric])):
            assert abs(ours.loc[arm, column] - saved[arm, metric]) < 1e-6, (arm, metric)
ours

,rmse,mae,bias,r2,rank
R0,1.9920,1.6401,0.1405,-0.0078,NaN
R1,1.8539,1.4339,-0.0693,0.1271,-0.0536
A1,1.7852,1.4485,0.0813,0.1905,0.2990


### 4.2 Time features: which summary of the monthly series works (`experiments/time_features/`)

Every variant is label-free: it uses only the field's own clear months. Same Ridge head as A1.

In [8]:
_, _, _, monthly = tf.load(COHORT, expected, labels)
variants, _ = tf.build(COHORT, X_a1, y, monthly, pre_months)
time_pred = {name: step8.out_of_fold(X, y, fields, ("0",))["A1"][scored] for name, X in variants.items()}
time_table = pd.DataFrame({name: {"features": variants[name].shape[1], **scores(yt, p, cluster_years[scored])}
                           for name, p in time_pred.items()}).T
saved = pd.read_csv(REPO / "experiments" / "time_features" / "results.csv").query("cohort == @COHORT").set_index("variant")
for name in time_table.index:
    for column, saved_column in (("rmse", "rmse"), ("mae", "mae"), ("bias", "bias"), ("r2", "r2"), ("rank", "rank_inside_clusters")):
        assert abs(time_table.loc[name, column] - saved.loc[name, saved_column]) < 1e-6, (name, column)
time_table.join(saved[["rank_vs_A1", "rank_lo", "rank_hi", "p_rank_holm", "wins"]])

,features,rmse,mae,bias,r2,rank,rank_vs_A1,rank_lo,rank_hi,p_rank_holm,wins
A1 max + mean,10.0,1.7852,1.4485,0.0813,0.1905,0.2990,NaN,NaN,NaN,NaN,False
"V0 mean, whole window",5.0,1.8571,1.5150,0.1225,0.1240,0.2616,-0.0374,-0.0722,-0.0026,0.319,False
"V1 mean, pre-harvest months",5.0,1.8256,1.4850,0.1077,0.1535,0.2958,-0.0032,-0.0371,0.0312,1.000,False
V2 max (peak value),5.0,1.7958,1.4570,0.0925,0.1809,0.3068,0.0078,-0.0110,0.0264,1.000,False
"V2+ peak, peak month, area before peak",9.0,1.7652,1.4236,0.0950,0.2086,0.3188,0.0198,-0.0179,0.0586,1.000,False
V3 mean + median + max,15.0,1.7843,1.4454,0.0815,0.1913,0.2997,0.0008,-0.0065,0.0079,1.000,False
L1 GCVI peak only (Li et al. 2022),1.0,1.8403,1.4992,0.1142,0.1398,0.2988,-0.0002,-0.0305,0.0324,1.000,False
C1 mean within 1 month of the peak,5.0,1.8034,1.4573,0.1127,0.1739,0.2930,-0.0060,-0.0466,0.0343,1.000,False
M1 monthly index curve,35.0,1.7331,1.3796,0.0760,0.2371,0.3270,0.0280,-0.0186,0.0789,1.000,False
"M1-pre monthly index curve, pre-harvest",25.0,1.7469,1.3990,0.0868,0.2249,0.3440,0.0450,0.0002,0.0894,0.480,False


### 4.3 Presto and the raw-band control

P0 is the frozen Presto embedding (128 numbers per field, no labels) with the same Ridge head; it is recomputed here
from the saved embeddings. X1 (Ridge on the 99 raw monthly inputs) and P1 (fine-tuned Presto) need `torch`, so
they are read from `experiments/presto_finetune/runs/zimbabwe_lacuna_2022_2023/posthoc_controls.csv`.

In [9]:
embeddings = np.load(REPO / "experiments" / "presto_finetune" / "runs" / COHORT / "p0_embeddings.npz", allow_pickle=True)
P0 = pd.DataFrame(embeddings["P0"], index=embeddings["obs_id"].astype(str)).reindex(y.index)
p0 = step8.out_of_fold(P0, y, fields, ("0",))["A1"][scored]
presto = pd.read_csv(REPO / "experiments" / "presto_finetune" / "runs" / COHORT / "posthoc_controls.csv")
presto_arms = presto[presto["kind"] == "arm"].set_index("arm")[["rmse", "mae", "bias", "r2", "rank_inside_clusters"]]
p0_live = pd.Series(scores(yt, p0, cluster_years[scored]), name="P0 recomputed here")
for column, saved_column in (("rmse", "rmse"), ("mae", "mae"), ("bias", "bias"), ("r2", "r2"), ("rank", "rank_inside_clusters")):
    assert abs(p0_live[column] - presto_arms.loc["P0", saved_column]) < 1e-6, column
display(presto_arms.rename(columns={"rank_inside_clusters": "rank"}))
presto[presto["kind"] == "pair"][["a", "b", "rmse_diff", "rmse_low", "rmse_high", "rank_diff", "rank_low", "rank_high"]]

,rmse,mae,bias,r2,rank
arm,,,,,
A1,1.7852,1.4485,0.0813,0.1905,0.2990
R1,1.8539,1.4339,-0.0693,0.1271,-0.0536
P0,1.6572,1.3032,0.0670,0.3024,0.3580
P1,1.6521,1.2898,0.0569,0.3067,0.3530
X1,1.6260,1.2681,0.0797,0.3284,0.3498
X2,1.6652,1.3156,0.0893,0.2957,0.3413


,a,b,rmse_diff,rmse_low,rmse_high,rank_diff,rank_low,rank_high
6,X1,A1,-0.1592,-0.2436,-0.0838,0.0508,-0.0044,0.1085
7,X2,A1,-0.1200,-0.1862,-0.0533,0.0424,-0.0078,0.0931
8,P0,X1,0.0312,-0.0162,0.0781,0.0083,-0.0286,0.0429
9,P0,X2,-0.0080,-0.0545,0.0407,0.0167,-0.0136,0.0475
10,P1,X1,0.0261,-0.0134,0.0652,0.0032,-0.0294,0.0320


### 4.4 The step-up ladder (`experiments/stepup/`)

From the simplest arm to monthly bands, Presto and MOSAIKS, Ridge head. **R² between** is how well the 2 km cluster
means are predicted, **R² within** how well the deviations from the cluster mean are. `p Holm` corrects the rank
p-value over every step except R1 (the bug that left S10–S12 out was fixed on 8 Nov). Read from `results.csv`.

In [10]:
ladder = pd.read_csv(REPO / "experiments" / "stepup" / "results.csv")
ladder = ladder[(ladder["cohort"] == COHORT) & ladder["head"].isin(["Ridge", "-", "kNN"])]
ladder[["step", "n_features", "rmse", "mae", "bias", "r2", "r2_between", "r2_within", "rank_inside_clusters",
        "train_r2", "rank_vs_A1", "rank_lo", "rank_hi", "p_rank_holm"]].set_index("step")

,n_features,rmse,mae,bias,r2,r2_between,r2_within,rank_inside_clusters,train_r2,rank_vs_A1,rank_lo,rank_hi,p_rank_holm
step,,,,,,,,,,,,,
S0 R0 training mean,0,1.9920,1.6401,0.1405,-0.0078,-0.0141,0.0000,NaN,NaN,NaN,NaN,NaN,NaN
S1 R1 coordinates only (10 nearest fields),2,1.8539,1.4339,-0.0693,0.1271,0.2505,-0.0276,-0.0536,NaN,-0.3526,-0.4327,-0.2599,NaN
S2 GCVI peak,1,1.8403,1.4992,0.1142,0.1398,0.1825,0.0862,0.2988,0.1634,-0.0002,-0.0305,0.0324,1.000
"S3 GCVI season shape (peak, mean, peak month, area before peak)",4,1.7949,1.4446,0.0987,0.1817,0.2692,0.0720,0.3245,0.2192,0.0255,-0.0135,0.0664,1.000
S4 GCVI monthly curve,7,1.7722,1.4313,0.1194,0.2023,0.2681,0.1198,0.3576,0.2427,0.0586,0.0137,0.1094,0.350
"S5 A1: 5 indices, max + mean",10,1.7852,1.4485,0.0813,0.1905,0.2990,0.0546,0.2990,0.2288,NaN,NaN,NaN,NaN
"S6 5 indices, monthly curve",35,1.7331,1.3796,0.0760,0.2371,0.3553,0.0890,0.3270,0.3186,0.0280,-0.0186,0.0789,1.000
"S7 10 bands, monthly curve",70,1.6338,1.2772,0.0788,0.3220,0.5020,0.0964,0.3419,0.4434,0.0429,-0.0152,0.1013,1.000
"S8 10 bands, monthly, pre-harvest months only",50,1.6393,1.2877,0.0696,0.3175,0.4932,0.0972,0.3426,0.4142,0.0437,-0.0100,0.0963,1.000


### 4.5 Codex's Presto debug runs (`experiments/presto_debug/`, 8 Nov)

A separate, smaller test of fine-tuning: one and two fields (can the model memorize?), then 128 training fields
(folds 2–4) and 64 validation fields (fold 1) with four levels of head dropout (0, 0.05, 0.2, 0.5) for 50 epochs.
Read from the saved `runs/latest/*/history.csv`; these numbers are not comparable with the 1,504-field table above.

In [11]:
debug = REPO / "experiments" / "presto_debug" / "runs" / "latest"
debug_rows = []
for stage in ("one", "two", "none", "light", "medium", "strong"):
    history = pd.read_csv(debug / stage / "history.csv")
    config = json.loads((debug / stage / "config.json").read_text(encoding="utf-8"))
    row = {"stage": stage, "fields": len(config["train_ids"]), "dropout": config["dropout"], "epochs run": int(history["epoch"].max()),
           "first train MSE": history["train_mse"].iloc[0], "last train MSE": history["train_mse"].iloc[-1]}
    if history["val_mse"].notna().any():
        best = history.loc[history["val_mse"].idxmin()]
        predictions = pd.read_csv(debug / stage / "predictions.csv")
        train_mean = predictions.loc[predictions["split"] == "train", "observed"].mean()
        validation = predictions.loc[predictions["split"] == "validation", "observed"]
        row |= {"best val MSE": best["val_mse"], "best epoch": int(best["epoch"]), "best val RMSE": np.sqrt(best["val_mse"]),
                "last val MSE": history["val_mse"].iloc[-1], "training-mean val MSE": float(np.mean((validation - train_mean) ** 2))}
    debug_rows.append(row)
debug_table = pd.DataFrame(debug_rows).set_index("stage")
debug_table

,fields,dropout,epochs run,first train MSE,last train MSE,best val MSE,best epoch,best val RMSE,last val MSE,training-mean val MSE
stage,,,,,,,,,,
one,1,0.00,13,8.9128,0.0036,NaN,NaN,NaN,NaN,NaN
two,2,0.00,100,12.1267,0.0067,NaN,NaN,NaN,NaN,NaN
none,128,0.00,50,4.3498,1.0370,2.5267,6.0,1.5895,3.8892,3.2707
light,128,0.05,50,4.3498,1.1055,2.5205,9.0,1.5876,3.8852,3.2707
medium,128,0.20,50,4.3498,1.2408,2.5206,14.0,1.5876,3.5698,3.2707
strong,128,0.50,50,4.3498,1.5757,2.4920,14.0,1.5786,2.9635,3.2707


## 5. Fold 0 (376 fields): train on folds 1–4, predict fold 0

The newest run of `experiments/fold0/score_fold0.py`, checked here by recomputing R0, R1, A1 and M2. Fold 0 is a
confirmation set, not an unseen test: development models trained on its fields, and R0, R1 and A1 were scored on it
before. `looks.csv` lists every look.

In [12]:
looks = pd.read_csv(REPO / "experiments" / "fold0" / "looks.csv")
latest = REPO / "experiments" / "fold0" / looks["run"].iloc[-1]
fold0 = pd.read_csv(latest / "scores.csv").set_index("arm")
fold0_pred = pd.read_csv(latest / "predictions.csv").set_index("obs_id")
test = (fields["fold"] == "0").to_numpy()
again = step8.out_of_fold(X_a1, y, fields, ("1", "2", "3", "4"))
groups = fields["group_id"].to_numpy()
m2 = variants["M2 monthly band curve"].to_numpy()
again["M2"] = np.full(len(y), np.nan)
again["M2"][test] = ridge_cv(m2[~test], y.to_numpy()[~test], groups[~test], m2[test], step8.ALPHAS)[0]
for arm in ("R0", "R1", "A1", "M2"):
    assert np.allclose(again[arm][test], fold0_pred.loc[fields.index[test], arm].to_numpy()), arm
print(f"{len(looks)} recorded look(s) at fold 0; newest: {looks['run'].iloc[-1]} ({looks['reason'].iloc[-1]})")
fold0[["name", "rmse", "mae", "bias", "r2", "rank_inside_clusters", "rank_vs_A1", "rank_lo", "rank_hi",
       "rmse_vs_A1", "rmse_lo", "rmse_hi", "p_rank_holm", "wins_vs_A1"]]

1 recorded look(s) at fold 0; newest: runs/20261108T162116Z (First repeatable fold-0 scoring of the main Lacuna arms after the owner ruling of 8 Nov (fold 0 may be scored more than once; every look is logged))


,name,rmse,mae,bias,r2,rank_inside_clusters,rank_vs_A1,rank_lo,rank_hi,rmse_vs_A1,rmse_lo,rmse_hi,p_rank_holm,wins_vs_A1
arm,,,,,,,,,,,,,,
R0,R0 training mean,2.0614,1.6438,-0.5620,-0.0803,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
R1,R1 coordinates only (10 nearest fields),1.8913,1.4643,-0.5323,0.0907,0.0867,-0.2742,-0.4184,-0.1334,0.1038,-0.1989,0.4149,NaN,False
A1,"A1 5 indices, season max + mean",1.7875,1.4016,-0.3508,0.1877,0.3610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
V0,"V0 mean, whole window",1.9467,1.5340,-0.4995,0.0366,0.2836,-0.0773,-0.1409,-0.0335,0.1592,0.0769,0.2413,0.0,False
V1,"V1 mean, pre-harvest months",1.8906,1.4918,-0.4653,0.0914,0.3420,-0.0190,-0.0830,0.0253,0.1031,0.0304,0.1792,1.0,False
V2,V2 max (peak value),1.8000,1.4038,-0.3823,0.1763,0.3852,0.0242,-0.0060,0.0534,0.0125,-0.0293,0.0469,1.0,False
V3,V3 mean + median + max,1.7803,1.3984,-0.3519,0.1942,0.3607,-0.0002,-0.0135,0.0078,-0.0071,-0.0179,0.0082,1.0,False
L1,L1 GCVI peak only (Li et al. 2022),1.8556,1.4546,-0.4695,0.1247,0.3454,-0.0155,-0.1041,0.0664,0.0681,0.0157,0.1137,1.0,False
M1,M1 monthly index curve,1.7521,1.3842,-0.3900,0.2196,0.3495,-0.0115,-0.0882,0.0367,-0.0354,-0.1242,0.0876,1.0,False


## 6. Felix's numbers

### 6.1 His v1 paper numbers against his rule-compliant v2 runs

`docs/paper/` (Felix, 7 Nov) quotes the v1 releases. On 8 Nov the v1 runs were found to break three team rules
(training on the 1,874 Zimbabwe 2024 zero yields and on 217 excluded Lacuna fields; a pre-harvest window anchored on
each field's planting and harvest dates; fold roles). They were re-run as v2. All rows: 1,504 Lacuna fields,
protocol B.

In [13]:
SOURCES = {"merge": ("data/runs/zimbabwe_merge_v1", "results/merge_v2/merge"),
           "diagnosis": ("data/runs/zimbabwe_transfer_diagnosis_v1", "results/merge_v2/diagnosis"),
           "weather": ("data/runs/zimbabwe_weather_v1", "results/weather_v2/full"),
           "compact": ("data/runs/zimbabwe_weather_compact_v1", "results/weather_v2/compact"),
           "aggregation": ("data/runs/zimbabwe_aggregation_v1", "results/aggregation_v2")}
HEADLINE = [("Basic index Ridge, Lacuna only", "merge", "lacuna_only", "lacuna_only", "none"),
            ("Basic index Ridge, naive pooling with 2024", "merge", "pooled_naive", "pooled_naive", "2024 zeros"),
            ("Basic index Ridge, adaptive pooling with 2024", "merge", "pooled_adaptive", "pooled_adaptive", "2024 zeros"),
            ("Monthly imagery Ridge, Lacuna only", "diagnosis", "monthly_target", "monthly_target", "none"),
            ("Monthly imagery trees, Lacuna only", "diagnosis", "monthly_tree_target", "monthly_tree_target", "none"),
            ("Monthly imagery trees, merged with 2024", "diagnosis", "monthly_tree_transfer", "monthly_tree_transfer", "2024 zeros"),
            ("Monthly + 46 weather, merged trees", "weather", "weather_tree_transfer", "weather_tree_transfer", "2024 zeros"),
            ("Monthly + 4 weather, shared Ridge", "compact", "compact4_shared", "compact4_shared", "2024 zeros"),
            ("Pre-harvest band means, Lacuna only", "aggregation", "bands_v1_preharvest_mean_target",
             "bands_v1_calendar_mean_target", "crop-date window (v2: Dec-Mar calendar)"),
            ("Season band means, Lacuna only", "aggregation", "bands_v0_mean_target", "bands_v0_mean_target", "none")]
metrics = {(step, v): pd.read_csv(REPO / path / "metrics.csv").set_index("method")
           for step, paths in SOURCES.items() for v, path in zip(("v1", "v2"), paths)}
felix = pd.DataFrame([{"model": name, "v1 rmse": metrics[step, "v1"].loc[m1, "rmse_t_ha"],
                       "v1 r2": metrics[step, "v1"].loc[m1, "r2"], "rule broken in v1": rule,
                       "v2 rmse": metrics[step, "v2"].loc[m2_, "rmse_t_ha"], "v2 mae": metrics[step, "v2"].loc[m2_, "mae_t_ha"],
                       "v2 bias": metrics[step, "v2"].loc[m2_, "bias_t_ha"], "v2 r2": metrics[step, "v2"].loc[m2_, "r2"]}
                      for name, step, m1, m2_, rule in HEADLINE]).set_index("model")
felix["change"] = felix["v2 rmse"] - felix["v1 rmse"]
felix

,v1 rmse,v1 r2,rule broken in v1,v2 rmse,v2 mae,v2 bias,v2 r2,change
model,,,,,,,,
"Basic index Ridge, Lacuna only",1.7968,0.1800,none,1.7968,1.4477,-0.0137,0.1800,0.0000
"Basic index Ridge, naive pooling with 2024",2.2707,-0.3096,2024 zeros,2.0858,1.5785,-1.0589,-0.1050,-0.1849
"Basic index Ridge, adaptive pooling with 2024",1.7938,0.1827,2024 zeros,1.7932,1.4457,-0.0202,0.1833,-0.0006
"Monthly imagery Ridge, Lacuna only",1.6926,0.2723,none,1.6926,1.3223,0.0045,0.2723,0.0000
"Monthly imagery trees, Lacuna only",1.6594,0.3006,none,1.6594,1.2897,-0.0054,0.3006,0.0000
"Monthly imagery trees, merged with 2024",1.6578,0.3020,2024 zeros,1.6770,1.3038,-0.0458,0.2857,0.0192
"Monthly + 46 weather, merged trees",1.7221,0.2467,2024 zeros,1.7164,1.3364,-0.0771,0.2517,-0.0057
"Monthly + 4 weather, shared Ridge",1.6554,0.3040,2024 zeros,1.6428,1.2763,-0.0498,0.3146,-0.0126
"Pre-harvest band means, Lacuna only",1.7503,0.2219,crop-date window (v2: Dec-Mar calendar),1.7540,1.4015,0.0129,0.2186,0.0037


In [14]:
runs = REPO / "data" / "runs" / "zimbabwe_merge_v2" / "predictions.csv"
if runs.exists():
    p = pd.read_csv(runs)
    again = pd.DataFrame({method: {"n": len(d), "rmse": rmse(d.yield_t_ha.to_numpy(), d.prediction_t_ha.to_numpy()),
                                   "r2": r2(d.yield_t_ha.to_numpy(), d.prediction_t_ha.to_numpy())}
                          for method, d in p.groupby("method")}).T
    saved = metrics["merge", "v2"]
    assert np.allclose(again["rmse"], saved.loc[again.index, "rmse_t_ha"], atol=1e-9)
    print("Felix's merge v2 metrics recomputed from his saved predictions: they match.")
    display(again)
else:
    print("data/runs/zimbabwe_merge_v2 is not on this machine (it is git-ignored); the table above uses the committed copies.")

Felix's merge v2 metrics recomputed from his saved predictions: they match.


,n,rmse,r2
lacuna_only,1504.0,1.7968,0.1800
lacuna_tree,1504.0,1.8026,0.1747
pooled_adaptive,1504.0,1.7932,0.1833
pooled_intercept,1504.0,1.7923,0.1841
pooled_naive,1504.0,2.0858,-0.1050
pooled_tree,1504.0,1.7948,0.1818
train_mean,1504.0,1.9881,-0.0039


### 6.2 How his Lacuna-only Ridge becomes ours, one switch at a time

Same model (Ridge on the 10 season indices), same 1,504 scored fields. Each row changes one setting.

In [15]:
split = pd.read_csv(V1 / "splits" / f"{COHORT}__s1_cc2km.csv", dtype={"fold": str, "group_id": str},
                    keep_default_na=False).set_index("obs_id")
ALPHAS_A, ALPHAS_B = step8.ALPHAS, np.array([0.01, 0.1, 1.0, 10.0, 100.0, 1000.0])


def a1_dev(database, fold0_trains, inner, alphas):
    X, target = load_xy(COHORT, database)
    f = split.loc[X.index]
    pred = np.full(len(target), np.nan)
    for k in ("1", "2", "3", "4"):
        test_k = f["fold"].eq(k).to_numpy()
        train = ~test_k if fold0_trains else (~test_k & f["fold"].ne("0").to_numpy())
        pred[test_k] = ridge_cv(X.to_numpy()[train], target.to_numpy()[train], f["group_id"].to_numpy()[train],
                                X.to_numpy()[test_k], alphas, inner)[0]
    keep = f["fold"].ne("0").to_numpy()
    return rmse(target.to_numpy()[keep], pred[keep]), r2(target.to_numpy()[keep], pred[keep])


chain = [("Ours (protocol A): v2, fold 0 trains, 5 inner folds, 17 alphas", V2, True, 5, ALPHAS_A),
         ("Switch the database to v1", V1, True, 5, ALPHAS_A),
         ("Keep fold 0 out of every fit", V1, False, 5, ALPHAS_A),
         ("Use 3 inner folds", V1, False, 3, ALPHAS_A),
         ("Use Felix's 6 alphas (protocol B)", V1, False, 3, ALPHAS_B)]
a1_chain = pd.DataFrame([dict(zip(("rmse", "r2"), a1_dev(*row[1:])), step=row[0]) for row in chain]).set_index("step")
a1_chain["change"] = a1_chain["rmse"].diff()
assert abs(a1_chain["rmse"].iloc[0] - ours.loc["A1", "rmse"]) < 1e-9
assert abs(a1_chain["rmse"].iloc[-1] - metrics["merge", "v2"].loc["lacuna_only", "rmse_t_ha"]) < 1e-6
print("Last row = Felix's lacuna_only in results/merge_v2/merge/metrics.csv:",
      round(metrics["merge", "v2"].loc["lacuna_only", "rmse_t_ha"], 6))
a1_chain

Last row = Felix's lacuna_only in results/merge_v2/merge/metrics.csv: 1.796767


,rmse,r2,change
step,,,
"Ours (protocol A): v2, fold 0 trains, 5 inner folds, 17 alphas",1.7852,0.1905,NaN
Switch the database to v1,1.7864,0.1895,0.0012
Keep fold 0 out of every fit,1.7940,0.1825,0.0076
Use 3 inner folds,1.7940,0.1825,0.0000
Use Felix's 6 alphas (protocol B),1.7968,0.1800,0.0027


### 6.3 Monthly features: his 122 inputs against our 70-input band curve, under both protocols

Felix's monthly Ridge (`monthly_target`) uses the 10 season indices, 15 variables x 7 months (Dec–Jun) with missing
months left empty, and 7 missing-month flags (122 inputs); the training fold fills gaps with its median. Our M2 uses
10 bands x 7 months, interpolated inside each field (70 inputs). The code below reproduces his head (median fill,
1–99 % clip, scaling, 3 inner folds, 6 alphas, pooled inner error, predictions clipped at 0) and ours, and runs both
feature sets under all 8 combinations of database (v1, v2), fold-0 rule (trains, out) and head (A = ours, B = his).
Protocol A is v2 + fold 0 trains + head A; protocol B is v1 + fold 0 out + head B.

In [16]:
BANDS, INDICES = tf.BANDS, tf.INDICES
SEASON = [f"{i}_season_{s}" for i in ("ndvi", "gcvi", "mtci", "ndvi705", "ndvi740") for s in ("max", "mean")]
ORDER = ["12", "01", "02", "03", "04", "05", "06"]


def monthly_inputs(database):
    with duckdb.connect(str(database / "cropyield.duckdb"), read_only=True) as con:
        rows = con.execute(f"""SELECT obs_id, yield_kg_ha / 1000 AS y, {', '.join(SEASON)} FROM master
                               WHERE cohort = ? AND include_primary AND months_with_data >= 4 ORDER BY obs_id""",
                           [COHORT]).df().set_index("obs_id")
        long = con.execute(f"""SELECT b.obs_id, b.month, b.missing, {', '.join('b.' + c for c in BANDS)},
                                      {', '.join('v.' + c for c in INDICES)}
                               FROM monthly_bands b JOIN vegetation_indices v USING (obs_id, month)
                               WHERE b.obs_id IN (SELECT unnest(?))""", [list(rows.index)]).df()
    rows = rows.join(split[["fold", "group_id"]])
    long = long[long["month"].str[5:].isin(ORDER)].assign(mm=lambda d: d["month"].str[5:])
    wide = long.pivot(index="obs_id", columns="mm", values=BANDS + INDICES).reindex(
        index=rows.index, columns=pd.MultiIndex.from_product([BANDS + INDICES, ORDER]))
    flags = long.pivot(index="obs_id", columns="mm", values="missing").astype(float).reindex(index=rows.index, columns=ORDER)
    temporal = wide.set_axis([f"{v}_month_{m}" for v, m in wide.columns], axis=1)
    f122 = pd.concat([rows[SEASON], temporal, flags.add_prefix("missing_month_")], axis=1)
    own = {b: wide[b].mean(axis=1) for b in BANDS}
    m2_inputs = pd.concat([wide[b].interpolate(axis=1, limit_direction="both").apply(lambda c, o=own[b]: c.fillna(o))
                           .add_prefix(f"{b}_m") for b in BANDS], axis=1)
    return rows, {"Felix F122 (median fill + flags)": f122, "Our M2 (interpolated)": m2_inputs}


class TrainingTransform:
    def fit(self, x):
        x = np.where(np.isfinite(x), x, np.nan)
        x[:, np.isnan(x).all(axis=0)] = 0.0
        self.low, self.median, self.high = np.nanquantile(x, [0.01, 0.5, 0.99], axis=0)
        filled = np.clip(np.where(np.isnan(x), self.median, x), self.low, self.high)
        self.mean, self.scale = filled.mean(axis=0), filled.std(axis=0)
        self.scale[self.scale < 1e-12] = 1.0
        return self

    def transform(self, x):
        x = np.where(np.isfinite(x), x, self.median)
        return (np.clip(x, self.low, self.high) - self.mean) / self.scale


def head_fit(x_train, y_train, g_train, x_test, head):
    inner, alphas, pooled, clip = (5, ALPHAS_A, False, False) if head == "A" else (3, ALPHAS_B, True, True)

    def one(alpha, a, b, c):
        t = TrainingTransform().fit(a)
        pred = Ridge(alpha=alpha).fit(t.transform(a), b).predict(t.transform(c))
        return np.maximum(pred, 0) if clip else pred

    splits = list(GroupKFold(inner).split(x_train, groups=g_train))
    losses = []
    for alpha in alphas:
        if pooled:
            pred = np.concatenate([one(alpha, x_train[f], y_train[f], x_train[v]) for f, v in splits])
            truth = np.concatenate([y_train[v] for _, v in splits])
            losses.append(np.mean((pred - truth) ** 2))
        else:
            losses.append(np.mean([np.mean((one(alpha, x_train[f], y_train[f], x_train[v]) - y_train[v]) ** 2)
                                   for f, v in splits]))
    return one(alphas[int(np.argmin(losses))], x_train, y_train, x_test)


def monthly_dev(rows, X, fold0_trains, head):
    target, fold, group, values = rows["y"].to_numpy(), rows["fold"].to_numpy(), rows["group_id"].to_numpy(), X.to_numpy(float)
    pred = np.full(len(target), np.nan)
    for k in ("1", "2", "3", "4"):
        test_k = fold == k
        train = ~test_k if fold0_trains else (~test_k & (fold != "0"))
        pred[test_k] = head_fit(values[train], target[train], group[train], values[test_k], head)
    return pred


F122, M2 = "Felix F122 (median fill + flags)", "Our M2 (interpolated)"
PROTOCOL_A, PROTOCOL_B = ("v2", True, "A"), ("v1", False, "B")
inputs = {version: monthly_inputs(folder) for version, folder in (("v1", V1), ("v2", V2))}
grid, monthly_pred = [], {}
for features in (F122, M2):
    for version in ("v1", "v2"):
        rows, feats = inputs[version]
        truth, keep = rows["y"].to_numpy(), (rows["fold"] != "0").to_numpy()
        for fold0_trains in (True, False):
            for head in ("A", "B"):
                pred = monthly_dev(rows, feats[features], fold0_trains, head)
                monthly_pred[features, version, fold0_trains, head] = pred
                grid.append({"features": features, "database": version, "fold 0 trains": fold0_trains, "head": head,
                             "rmse": rmse(truth[keep], pred[keep]), "r2": r2(truth[keep], pred[keep])})
monthly_grid = pd.DataFrame(grid).set_index(["features", "database", "fold 0 trains", "head"])
assert abs(monthly_grid.loc[(F122, *PROTOCOL_B), "rmse"] - metrics["diagnosis", "v2"].loc["monthly_target", "rmse_t_ha"]) < 1e-6
assert abs(monthly_grid.loc[(M2, *PROTOCOL_A), "rmse"] - time_table.loc["M2 monthly band curve", "rmse"]) < 1e-6
display(monthly_grid.unstack("features"))

rmse                                                     r2                      
features                    Felix F122 (median fill + flags) Our M2 (interpolated) Felix F122 (median fill + flags) Our M2 (interpolated)
database fold 0 trains head                                                                                                              
v1       False         A                              1.6742                1.6719                           0.2880                0.2901
                       B                              1.6926                1.6718                           0.2723                0.2901
         True          A                              1.6383                1.6371                           0.3183                0.3193
                       B                              1.6377                1.6371                           0.3188                0.3193
v2       False         A                              1.6620                1.6614                           0.2984                0.2989
                       B                              1.6817                1.6640                           0.2817                0.2967
         True          A                              1.6257                1.6338                           0.3287                0.3220
                       B                              1.6268                1.6301                           0.3278                0.3251

Each switch below changes one setting and holds the other two fixed; the range runs over the four ways to fix them.
Then the two feature sets are compared in pairs (same fields, same folds, paired cluster bootstrap, 2,000 draws).

In [17]:
from cropyield.eval.scores import paired_difference_ci

g = monthly_grid["rmse"]
switches = []
for features in (F122, M2):
    for version in ("v1", "v2"):
        for head in ("A", "B"):
            switches.append({"switch": "keep fold 0 out of training", "features": features,
                             "change": g[features, version, False, head] - g[features, version, True, head]})
        for trains in (True, False):
            switches.append({"switch": "Felix head instead of ours", "features": features,
                             "change": g[features, version, trains, "B"] - g[features, version, trains, "A"]})
    for trains in (True, False):
        for head in ("A", "B"):
            switches.append({"switch": "v1 database instead of v2", "features": features,
                             "change": g[features, "v1", trains, head] - g[features, "v2", trains, head]})
switch_range = pd.DataFrame(switches).groupby(["switch", "features"])["change"].agg(["min", "max"])
display(switch_range)

m2_vs_f122 = []
for label, protocol in (("protocol A", PROTOCOL_A), ("protocol B", PROTOCOL_B)):
    rows = inputs[protocol[0]][0]
    keep = (rows["fold"] != "0").to_numpy()
    truth, groups_k = rows["y"].to_numpy()[keep], rows["group_id"].to_numpy()[keep]
    a, b = monthly_pred[(M2, *protocol)][keep], monthly_pred[(F122, *protocol)][keep]
    low, high = paired_difference_ci(truth, a, b, groups_k, rmse, seed=step8.SEED)
    m2_vs_f122.append({"protocol": label, "M2 minus F122, RMSE": rmse(truth, a) - rmse(truth, b), "low": low, "high": high})
m2_vs_f122 = pd.DataFrame(m2_vs_f122).set_index("protocol")
m2_vs_f122

min     max
switch                      features                                        
Felix head instead of ours  Felix F122 (median fill + flags) -0.0006  0.0197
                            Our M2 (interpolated)            -0.0038  0.0026
keep fold 0 out of training Felix F122 (median fill + flags)  0.0360  0.0549
                            Our M2 (interpolated)             0.0275  0.0347
v1 database instead of v2   Felix F122 (median fill + flags)  0.0109  0.0126
                            Our M2 (interpolated)             0.0033  0.0105

,"M2 minus F122, RMSE",low,high
protocol,,,
protocol A,0.0082,-0.0230,0.0398
protocol B,-0.0208,-0.0546,0.0138


## 7. One table

Every row is one model on the 1,504 Lacuna fields (fold 0: 376). The protocol is in the column name, so no row mixes
protocols. Built from the numbers computed above.

In [18]:
fb = metrics["diagnosis", "v2"]
row = lambda dev_a, dev_b, f0, note: {"protocol A dev R2": dev_a, "protocol B dev R2": dev_b, "fold 0 R2": f0, "note": note}
consensus = pd.DataFrame({
    "R1 coordinates only": row(ours.loc["R1", "r2"], np.nan, fold0.loc["R1", "r2"], "place alone"),
    "A1 10 season indices, Ridge": row(ours.loc["A1", "r2"], a1_chain["r2"].iloc[-1], fold0.loc["A1", "r2"],
                                       "the reference arm"),
    "M2 monthly band curve, Ridge": row(time_table.loc["M2 monthly band curve", "r2"], monthly_grid.loc[(M2, *PROTOCOL_B), "r2"],
                                        fold0.loc["M2", "r2"], "best family on both protocols"),
    "Felix F122 monthly Ridge": row(monthly_grid.loc[(F122, *PROTOCOL_A), "r2"], fb.loc["monthly_target", "r2"], np.nan,
                                    "within the paired interval of M2 on both protocols"),
    "X1 raw monthly inputs, Ridge": row(presto_arms.loc["X1", "r2"], np.nan, fold0.loc["X1", "r2"], "the Presto control"),
    "P0 frozen Presto, Ridge": row(presto_arms.loc["P0", "r2"], np.nan, fold0.loc["P0", "r2"], "no gain over X1"),
    "Felix monthly trees, Lacuna only": row(np.nan, fb.loc["monthly_tree_target", "r2"], np.nan, "his best imagery-only arm"),
    "Felix 4 weather, shared Ridge (+2024)": row(np.nan, metrics["compact", "v2"].loc["compact4_shared", "r2"], np.nan,
                                                 "pooled exception; gain over trees not established"),
}).T
consensus

,protocol A dev R2,protocol B dev R2,fold 0 R2,note
R1 coordinates only,0.1271,NaN,0.0907,place alone
"A1 10 season indices, Ridge",0.1905,0.18,0.1877,the reference arm
"M2 monthly band curve, Ridge",0.322,0.2901,0.3956,best family on both protocols
Felix F122 monthly Ridge,0.3287,0.2723,NaN,within the paired interval of M2 on both protocols
"X1 raw monthly inputs, Ridge",0.3284,NaN,0.3522,the Presto control
"P0 frozen Presto, Ridge",0.3024,NaN,0.3232,no gain over X1
"Felix monthly trees, Lacuna only",NaN,0.3006,NaN,his best imagery-only arm
"Felix 4 weather, shared Ridge (+2024)",NaN,0.3146,NaN,pooled exception; gain over trees not established


In [19]:
pairs = presto[presto["kind"] == "pair"].set_index(["a", "b"])
ladder_by = ladder[ladder["head"] == "Ridge"].set_index("step")
aggregation_pairs = pd.read_csv(REPO / "results" / "aggregation_v2" / "paired_differences.csv").set_index(["method", "reference"])
merge_pairs = pd.read_csv(REPO / "results" / "merge_v2" / "merge" / "paired_differences.csv").set_index(["method", "reference"])
candidate = pd.read_csv(REPO / "results" / "weather_v2" / "compact_candidate.csv")
candidate = candidate[candidate["bootstrap_unit"] == "spatial_group"].set_index("reference")
compact_pairs = pd.read_csv(REPO / "results" / "weather_v2" / "compact" / "paired_differences.csv")
compact_pairs = compact_pairs[compact_pairs["bootstrap_unit"] == "spatial_group"].set_index(["method", "reference"])
QUOTED = {
    "M2 R2, protocol A": (time_table.loc["M2 monthly band curve", "r2"], 0.322),
    "M2 R2, protocol B": (monthly_grid.loc[(M2, *PROTOCOL_B), "r2"], 0.290),
    "A1 R2": (ours.loc["A1", "r2"], 0.191),
    "R1 R2": (ours.loc["R1", "r2"], 0.127),
    "fold 0: M2 R2": (fold0.loc["M2", "r2"], 0.396),
    "fold 0: M2 minus A1, RMSE": (fold0.loc["M2", "rmse_vs_A1"], -0.246),
    "fold 0: low": (fold0.loc["M2", "rmse_lo"], -0.363),
    "fold 0: high": (fold0.loc["M2", "rmse_hi"], -0.089),
    "Felix monthly Ridge R2, protocol A": (monthly_grid.loc[(F122, *PROTOCOL_A), "r2"], 0.329),
    "M2 minus F122, protocol A": (m2_vs_f122.loc["protocol A", "M2 minus F122, RMSE"], 0.008),
    "M2 minus F122, protocol A, low": (m2_vs_f122.loc["protocol A", "low"], -0.023),
    "M2 minus F122, protocol A, high": (m2_vs_f122.loc["protocol A", "high"], 0.040),
    "M2 minus F122, protocol B": (m2_vs_f122.loc["protocol B", "M2 minus F122, RMSE"], -0.021),
    "M2 minus F122, protocol B, low": (m2_vs_f122.loc["protocol B", "low"], -0.055),
    "M2 minus F122, protocol B, high": (m2_vs_f122.loc["protocol B", "high"], 0.014),
    "fold 0 out, M2, smallest": (switch_range.loc[("keep fold 0 out of training", M2), "min"], 0.028),
    "fold 0 out, M2, largest": (switch_range.loc[("keep fold 0 out of training", M2), "max"], 0.035),
    "fold 0 out, F122, smallest": (switch_range.loc[("keep fold 0 out of training", F122), "min"], 0.036),
    "fold 0 out, F122, largest": (switch_range.loc[("keep fold 0 out of training", F122), "max"], 0.055),
    "head, largest change": (switch_range.loc["Felix head instead of ours"].abs().to_numpy().max(), 0.020),
    "database, largest change": (switch_range.loc["v1 database instead of v2"].abs().to_numpy().max(), 0.013),
    "Felix monthly Ridge R2, protocol B": (fb.loc["monthly_target", "r2"], 0.272),
    "P0 R2": (presto_arms.loc["P0", "r2"], 0.302),
    "X1 R2": (presto_arms.loc["X1", "r2"], 0.328),
    "P1 R2": (presto_arms.loc["P1", "r2"], 0.307),
    "P0 minus X1": (pairs.loc[("P0", "X1"), "rmse_diff"], 0.031),
    "P0 minus X1, low": (pairs.loc[("P0", "X1"), "rmse_low"], -0.016),
    "P0 minus X1, high": (pairs.loc[("P0", "X1"), "rmse_high"], 0.078),
    "smallest ladder Holm p, Ridge head": (ladder["p_rank_holm"].min(), 0.288),
    "smallest ladder Holm p, any head": (pd.read_csv(REPO / "experiments" / "stepup" / "results.csv")
                                         .query("cohort == @COHORT")["p_rank_holm"].min(), 0.222),
    "Dec-Mar bands minus season bands": (aggregation_pairs.loc[("bands_v1_calendar_mean_target", "bands_v0_mean_target"), "delta_rmse_t_ha"], -0.044),
    "Dec-Mar, low": (aggregation_pairs.loc[("bands_v1_calendar_mean_target", "bands_v0_mean_target"), "ci_low"], -0.081),
    "Dec-Mar, high": (aggregation_pairs.loc[("bands_v1_calendar_mean_target", "bands_v0_mean_target"), "ci_high"], -0.009),
    "M2 R2 between clusters": (ladder_by.loc["S7 10 bands, monthly curve", "r2_between"], 0.502),
    "M2 R2 within clusters": (ladder_by.loc["S7 10 bands, monthly curve", "r2_within"], 0.096),
    "A1 R2 between clusters": (ladder_by.loc["S5 A1: 5 indices, max + mean", "r2_between"], 0.299),
    "A1 R2 within clusters": (ladder_by.loc["S5 A1: 5 indices, max + mean", "r2_within"], 0.055),
    "A1 rank": (ours.loc["A1", "rank"], 0.299),
    "M2 rank": (ladder_by.loc["S7 10 bands, monthly curve", "rank_inside_clusters"], 0.342),
    "adaptive pooling minus Lacuna only": (merge_pairs.loc[("pooled_adaptive", "lacuna_only"), "delta_rmse_t_ha"], -0.0036),
    "adaptive, low": (merge_pairs.loc[("pooled_adaptive", "lacuna_only"), "ci_low"], -0.0084),
    "adaptive, high": (merge_pairs.loc[("pooled_adaptive", "lacuna_only"), "ci_high"], 0.0007),
    "naive pooling RMSE": (metrics["merge", "v2"].loc["pooled_naive", "rmse_t_ha"], 2.086),
    "trees with 2024 minus Lacuna-only trees": (compact_pairs.loc[("monthly_tree_transfer", "monthly_tree_target"), "delta_rmse_t_ha"], 0.018),
    "trees, low": (compact_pairs.loc[("monthly_tree_transfer", "monthly_tree_target"), "ci_low"], -0.007),
    "trees, high": (compact_pairs.loc[("monthly_tree_transfer", "monthly_tree_target"), "ci_high"], 0.045),
    "4 weather: shared minus Lacuna-only twin": (compact_pairs.loc[("compact4_shared", "compact4_target"), "delta_rmse_t_ha"], -0.086),
    "twin, low": (compact_pairs.loc[("compact4_shared", "compact4_target"), "ci_low"], -0.140),
    "twin, high": (compact_pairs.loc[("compact4_shared", "compact4_target"), "ci_high"], -0.041),
    "4 weather Lacuna-only twin RMSE": (metrics["compact", "v2"].loc["compact4_target", "rmse_t_ha"], 1.728),
    "Lacuna-only trees RMSE": (fb.loc["monthly_tree_target", "rmse_t_ha"], 1.659),
    "4 weather shared Ridge RMSE": (metrics["compact", "v2"].loc["compact4_shared", "rmse_t_ha"], 1.643),
    "weather minus trees": (candidate.loc["monthly_tree_target", "delta_rmse_t_ha"], -0.017),
    "weather, low": (candidate.loc["monthly_tree_target", "ci_low"], -0.058),
    "weather, high": (candidate.loc["monthly_tree_target", "ci_high"], 0.027),
    "fold 0: training-mean bias": (fold0.loc["R0", "bias"], -0.562),
    "Felix Lacuna-only Ridge": (a1_chain["rmse"].iloc[-1], 1.797),
    "our A1": (a1_chain["rmse"].iloc[0], 1.785),
    "switch: v1 database": (a1_chain["change"].iloc[1], 0.001),
    "debug: earliest best epoch": (debug_table["best epoch"].min(), 6),
    "debug: latest best epoch": (debug_table["best epoch"].max(), 14),
    "debug: lowest best val RMSE": (debug_table["best val RMSE"].min(), 1.58),
    "debug: highest best val RMSE": (debug_table["best val RMSE"].max(), 1.59),
    "switch: fold 0 out": (a1_chain["change"].iloc[2], 0.008),
    "switch: alpha grid": (a1_chain["change"].iloc[4], 0.003),
}
quoted = pd.DataFrame([{"claim": name, "computed": value, "quoted": text} for name, (value, text) in QUOTED.items()])
quoted["digits"] = quoted["quoted"].map(lambda v: len(f"{v}".split(".")[1]) if "." in f"{v}" else 0)
quoted["ok"] = [abs(c - q) <= 0.5 * 10 ** -d + 1e-12 for c, q, d in zip(quoted["computed"], quoted["quoted"], quoted["digits"])]
assert quoted["ok"].all(), quoted[~quoted["ok"]]
print(f"All {len(quoted)} numbers quoted in 'The answer' match the computed values.")
quoted.drop(columns="digits")

All 65 numbers quoted in 'The answer' match the computed values.


,claim,computed,quoted,ok
0,"M2 R2, protocol A",0.3220,0.322,True
1,"M2 R2, protocol B",0.2901,0.290,True
2,A1 R2,0.1905,0.191,True
3,R1 R2,0.1271,0.127,True
4,fold 0: M2 R2,0.3956,0.396,True
...,...,...,...,...
60,debug: latest best epoch,14.0000,14.000,True
61,debug: lowest best val RMSE,1.5786,1.580,True
62,debug: highest best val RMSE,1.5895,1.590,True
63,switch: fold 0 out,0.0076,0.008,True


## 8. Open points

- **Fold 0 in training.** Protocol A and protocol B are both written in `docs/EVALUATION.md`. The team must pick one
  before any table mixes the two lines of work.
- **v2 cloudy months.** Data v2 added 205 Lacuna months from scenes with a median tile cloud of 56 %. Felix's
  cloud review never checked them; A1 ranks slightly worse on v2 than on v1 (`results/v2/data_v2_comparison.md`).
- **Felix's `docs/paper/`** quotes the v1 numbers of section 6.1; it must be rebuilt from the v2 runs.
- **Fold 0** is a confirmation set (see section 5), and every new look at it is one more test of the same fields.